# Notebook 02: Data Cleaning & Merging

This notebook takes the raw data collected in Notebook 01 and produces a single
**master dataset** with one row per calendar day, containing oil prices, stock data,
and aggregated post features.

**Output**: `data/processed/master_dataset.csv` — 450+ rows, 15+ columns

In [1]:
import pandas as pd
import numpy as np
import os
import re

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)

## 1. Load Raw Data

In [2]:
# Load Truth Social posts
posts_raw = pd.read_csv('../data/raw/truth_posts.csv')
print(f'Posts: {len(posts_raw)} rows')

# Load oil prices
brent_raw = pd.read_csv('../data/raw/brent_daily.csv')
wti_raw = pd.read_csv('../data/raw/wti_daily.csv')
print(f'Brent: {len(brent_raw)} rows')
print(f'WTI: {len(wti_raw)} rows')

# Load DJT stock
djt_raw = pd.read_csv('../data/raw/djt_stock.csv')
print(f'DJT: {len(djt_raw)} rows')

Posts: 21014 rows
Brent: 313 rows
WTI: 313 rows
DJT: 306 rows


## 2. Clean Truth Social Posts

Parse timestamps, classify posts by topic relevance, and extract features.

In [3]:
def parse_timestamp(ts_str):
    """Parse trumpstruth.org timestamp strings into datetime."""
    if pd.isna(ts_str) or not ts_str:
        return pd.NaT
    
    # Format: 'March 24, 2026, 1:48 AM' or 'March 24, 2026, 12:30 PM'
    formats = [
        '%B %d, %Y, %I:%M %p',
        '%B %d, %Y, %I:%M%p',
        '%B %d, %Y',
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(ts_str.strip(), format=fmt)
        except (ValueError, AttributeError):
            continue
    
    # Fallback: let pandas figure it out
    try:
        return pd.to_datetime(ts_str)
    except:
        return pd.NaT

# Parse timestamps
posts_raw['timestamp'] = posts_raw['timestamp_str'].apply(parse_timestamp)

# Check parsing success rate
parsed = posts_raw['timestamp'].notna().sum()
print(f'Parsed {parsed}/{len(posts_raw)} timestamps ({parsed/len(posts_raw)*100:.1f}%)')

# Drop rows with unparseable timestamps
posts = posts_raw.dropna(subset=['timestamp']).copy()
posts = posts.sort_values('timestamp').reset_index(drop=True)

# Extract date and time features
posts['date'] = posts['timestamp'].dt.date
posts['hour'] = posts['timestamp'].dt.hour
posts['is_premarket'] = posts['hour'].apply(lambda h: 1 if h < 9 or (h == 9 and True) else 0)
# Note: Truth Social timestamps appear to be ET

print(f'\nDate range: {posts["date"].min()} to {posts["date"].max()}')
posts.head()

Parsed 21014/21014 timestamps (100.0%)

Date range: 2023-06-18 to 2026-03-24


,post_id,timestamp_str,post_text,post_url,original_url,image_descriptions,timestamp,date,hour,is_premarket
0,5437,"June 18, 2023, 9:56 AM",Believe,https://trumpstruth.org/statuses/5437,https://truthsocial.com/@JonVoight/110565563516545001,NaN,2023-06-18 09:56:00,2023-06-18,9,1
1,5147,"August 8, 2023, 3:28 PM",The Biden economy is not working. The American people were simply better off...,https://trumpstruth.org/statuses/5147,https://truthsocial.com/@garrettventry/110855644916610431,NaN,2023-08-08 15:28:00,2023-08-08,15,0
2,4995,"September 20, 2023, 7:56 PM",#IStandWithTrump@realDonaldTrump,https://trumpstruth.org/statuses/4995,https://truthsocial.com/@FruitSnacks/111100179649492292,"Donald Trump saluting in front of military personnel, with the text ""I Stand...",2023-09-20 19:56:00,2023-09-20,19,0
3,4995,"September 20, 2023, 7:56 PM",NaN,https://trumpstruth.org/statuses/4995,https://truthsocial.com/@realDonaldTrump/111922789349067805,"Donald Trump saluting in front of military personnel, with the text ""I Stand...",2023-09-20 19:56:00,2023-09-20,19,0
4,4997,"September 25, 2023, 11:11 AM",NaN,https://trumpstruth.org/statuses/4997,https://truthsocial.com/@realDonaldTrump/111922787041545495,"An image of a serious-looking man in a suit and tie, with an inset photo of ...",2023-09-25 11:11:00,2023-09-25,11,0


In [4]:
# Keyword-based oil/energy/geopolitical relevance classifier
OIL_KEYWORDS = [
    'oil', 'crude', 'energy', 'drill', 'drilling', 'petroleum', 'barrel',
    'opec', 'gas', 'gasoline', 'lng', 'pipeline', 'refinery',
    'iran', 'iranian', 'hormuz', 'strait', 'persian gulf',
    'sanctions', 'embargo', 'tariff',
    'nuclear', 'military', 'strike', 'bomb', 'obliterate', 'attack',
    'war', 'peace', 'ceasefire', 'surrender', 'negotiate', 'deal',
    'ultimatum', 'deadline', 'threat', 'retaliate',
    'china', 'chinese', 'trade war', 'liberation day'
]

def is_oil_related(text):
    """Check if post text contains oil/energy/geopolitical keywords."""
    if pd.isna(text):
        return False
    text_lower = text.lower()
    return any(kw in text_lower for kw in OIL_KEYWORDS)

def count_keyword_hits(text):
    """Count how many keyword categories are hit (crude measure of relevance)."""
    if pd.isna(text):
        return 0
    text_lower = text.lower()
    return sum(1 for kw in OIL_KEYWORDS if kw in text_lower)

posts['is_oil_related'] = posts['post_text'].apply(is_oil_related)
posts['keyword_count'] = posts['post_text'].apply(count_keyword_hits)

oil_posts = posts[posts['is_oil_related']]
print(f'Oil/energy/geopolitical posts: {len(oil_posts)} / {len(posts)} ({len(oil_posts)/len(posts)*100:.1f}%)')

Oil/energy/geopolitical posts: 3496 / 21014 (16.6%)


In [5]:
# Simple keyword-based direction classifier (before Gemini in Notebook 04)
ESCALATION_WORDS = [
    'obliterate', 'bomb', 'strike', 'destroy', 'war', 'attack', 'retaliate',
    'harder', 'ultimatum', 'surrender', 'threat', 'sanctions', 'embargo',
    'no deal', 'unconditional'
]

DEESCALATION_WORDS = [
    'peace', 'deal', 'negotiate', 'conversation', 'productive', 'agree',
    'winding down', 'complete', 'extension', 'ceasefire', 'resolution',
    'scaling back', 'talks'
]

def classify_direction(text):
    """Simple keyword-based direction classification."""
    if pd.isna(text):
        return 'neutral'
    text_lower = text.lower()
    esc = sum(1 for w in ESCALATION_WORDS if w in text_lower)
    deesc = sum(1 for w in DEESCALATION_WORDS if w in text_lower)
    
    if esc > deesc:
        return 'escalation'
    elif deesc > esc:
        return 'de_escalation'
    return 'neutral'

posts['direction_keyword'] = posts['post_text'].apply(classify_direction)

print('Direction classification (keyword-based):')
print(posts[posts['is_oil_related']]['direction_keyword'].value_counts())

Direction classification (keyword-based):
direction_keyword
de_escalation    1186
neutral          1160
escalation       1150
Name: count, dtype: int64


## 3. Clean Oil Price Data

In [6]:
# Clean Brent
brent = brent_raw.copy()
brent.columns = ['date', 'brent_close']
brent['date'] = pd.to_datetime(brent['date'])
brent['brent_close'] = pd.to_numeric(brent['brent_close'], errors='coerce')
brent = brent.dropna(subset=['brent_close'])
brent['date'] = brent['date'].dt.date

# Clean WTI
wti = wti_raw.copy()
wti.columns = ['date', 'wti_close']
wti['date'] = pd.to_datetime(wti['date'])
wti['wti_close'] = pd.to_numeric(wti['wti_close'], errors='coerce')
wti = wti.dropna(subset=['wti_close'])
wti['date'] = wti['date'].dt.date

# Clean DJT
djt = djt_raw.copy()
djt['date'] = pd.to_datetime(djt['date']).dt.date
djt['djt_close'] = pd.to_numeric(djt['djt_close'], errors='coerce')

print(f'Brent: {len(brent)} trading days')
print(f'WTI: {len(wti)} trading days')
print(f'DJT: {len(djt)} trading days')

Brent: 305 trading days
WTI: 298 trading days
DJT: 306 trading days


## 4. Aggregate Posts to Daily Level

In [7]:
def aggregate_daily_posts(posts_df):
    """Aggregate post-level data to one row per day."""
    daily = posts_df.groupby('date').agg(
        post_count=('post_text', 'count'),
        oil_post_count=('is_oil_related', 'sum'),
        keyword_hits_total=('keyword_count', 'sum'),
        escalation_count=('direction_keyword', lambda x: (x == 'escalation').sum()),
        deescalation_count=('direction_keyword', lambda x: (x == 'de_escalation').sum()),
        first_post_hour=('hour', 'min'),
        premarket_posts=('is_premarket', 'sum'),
    ).reset_index()
    
    # Direction balance: positive = more escalation, negative = more de-escalation
    daily['direction_balance'] = daily['escalation_count'] - daily['deescalation_count']
    
    # Dominant direction for the day
    daily['dominant_direction'] = daily['direction_balance'].apply(
        lambda x: 'escalation' if x > 0 else ('de_escalation' if x < 0 else 'neutral')
    )
    
    return daily

daily_posts = aggregate_daily_posts(posts)
print(f'Daily post features: {len(daily_posts)} days with posts')
daily_posts.head()

Daily post features: 855 days with posts


,date,post_count,oil_post_count,keyword_hits_total,escalation_count,deescalation_count,first_post_hour,premarket_posts,direction_balance,dominant_direction
0,2023-06-18,1,0,0,0,0,9,1,0,neutral
1,2023-08-08,1,0,0,0,0,15,0,0,neutral
2,2023-09-20,1,0,0,0,0,19,0,0,neutral
3,2023-09-25,1,0,0,0,0,11,0,0,neutral
4,2023-09-27,0,0,0,0,0,22,0,0,neutral


## 5. Build Master Dataset

Create a date spine covering every calendar day, then merge all data sources.

In [8]:
import datetime

# Date spine: every calendar day from Jan 1, 2025 to March 24, 2026
date_range = pd.date_range(start='2025-01-01', end='2026-03-24', freq='D')
master = pd.DataFrame({'date': date_range.date})

print(f'Date spine: {len(master)} calendar days')

# Merge daily post features (left join — days with no posts get 0)
master = master.merge(daily_posts, on='date', how='left')
master['post_count'] = master['post_count'].fillna(0).astype(int)
master['oil_post_count'] = master['oil_post_count'].fillna(0).astype(int)
master['keyword_hits_total'] = master['keyword_hits_total'].fillna(0).astype(int)
master['escalation_count'] = master['escalation_count'].fillna(0).astype(int)
master['deescalation_count'] = master['deescalation_count'].fillna(0).astype(int)
master['direction_balance'] = master['direction_balance'].fillna(0).astype(int)
master['premarket_posts'] = master['premarket_posts'].fillna(0).astype(int)
master['dominant_direction'] = master['dominant_direction'].fillna('no_posts')

# Merge Brent prices
master = master.merge(brent, on='date', how='left')

# Merge WTI prices
master = master.merge(wti, on='date', how='left')

# Merge DJT stock
master = master.merge(djt[['date', 'djt_close', 'djt_volume']], on='date', how='left')

print(f'Master dataset: {len(master)} rows, {len(master.columns)} columns')
master.head()

Date spine: 448 calendar days
Master dataset: 448 rows, 14 columns


,date,post_count,oil_post_count,keyword_hits_total,escalation_count,deescalation_count,first_post_hour,premarket_posts,direction_balance,dominant_direction,brent_close,wti_close,djt_close,djt_volume
0,2025-01-01,5,1,1,0,0,10.0,0,0,neutral,NaN,NaN,NaN,NaN
1,2025-01-02,8,3,3,3,1,0.0,6,2,escalation,76.14,73.79,34.020000,5166600.0
2,2025-01-03,31,5,6,2,1,0.0,23,1,escalation,76.72,74.64,34.619999,6150400.0
3,2025-01-04,2,1,2,1,0,7.0,2,1,escalation,NaN,NaN,NaN,NaN
4,2025-01-05,4,1,2,0,0,12.0,0,0,neutral,NaN,NaN,NaN,NaN


In [9]:
# Compute derived features

# Percentage changes (forward-fill prices first for continuity)
master['brent_close_ffill'] = master['brent_close'].ffill()
master['wti_close_ffill'] = master['wti_close'].ffill()
master['djt_close_ffill'] = master['djt_close'].ffill()

master['brent_pct_change'] = master['brent_close_ffill'].pct_change() * 100
master['wti_pct_change'] = master['wti_close_ffill'].pct_change() * 100
master['djt_pct_change'] = master['djt_close_ffill'].pct_change() * 100

# Brent-WTI spread
master['brent_wti_spread'] = master['brent_close'] - master['wti_close']

# 5-day rolling volatility (standard deviation of returns)
master['brent_volatility_5d'] = master['brent_pct_change'].rolling(5).std()

# Is it a trading day?
master['is_trading_day'] = master['brent_close'].notna().astype(int)

# Day of week (0=Monday, 6=Sunday)
master['day_of_week'] = pd.to_datetime(master['date']).dt.dayofweek

# Has oil-related post flag
master['has_oil_post'] = (master['oil_post_count'] > 0).astype(int)

print(f'Final master dataset: {len(master)} rows, {len(master.columns)} columns')
print(f'\nColumn types:')
print(master.dtypes)

Final master dataset: 448 rows, 25 columns

Column types:
date                    object
post_count               int64
oil_post_count           int64
keyword_hits_total       int64
escalation_count         int64
deescalation_count       int64
first_post_hour        float64
premarket_posts          int64
direction_balance        int64
dominant_direction      object
brent_close            float64
wti_close              float64
djt_close              float64
djt_volume             float64
brent_close_ffill      float64
wti_close_ffill        float64
djt_close_ffill        float64
brent_pct_change       float64
wti_pct_change         float64
djt_pct_change         float64
brent_wti_spread       float64
brent_volatility_5d    float64
is_trading_day           int64
day_of_week              int32
has_oil_post             int64
dtype: object


In [10]:
# Verify we meet requirements: 300+ rows, 12+ features
print('=== DATASET REQUIREMENTS CHECK ===')
print(f'Rows: {len(master)} (need 300+) {"PASS" if len(master) >= 300 else "FAIL"}')
print(f'Columns: {len(master.columns)} (need 12+) {"PASS" if len(master.columns) >= 12 else "FAIL"}')
print()

# Check feature types
numeric_cols = master.select_dtypes(include=[np.number]).columns
categorical_cols = master.select_dtypes(include=['object']).columns
print(f'Numeric features: {len(numeric_cols)}')
for c in numeric_cols:
    print(f'  - {c}')
print(f'\nCategorical features: {len(categorical_cols)}')
for c in categorical_cols:
    print(f'  - {c}')

print(f'\nMixed types: {"PASS" if len(numeric_cols) > 0 and len(categorical_cols) > 0 else "FAIL"}')

=== DATASET REQUIREMENTS CHECK ===
Rows: 448 (need 300+) PASS
Columns: 25 (need 12+) PASS

Numeric features: 23
  - post_count
  - oil_post_count
  - keyword_hits_total
  - escalation_count
  - deescalation_count
  - first_post_hour
  - premarket_posts
  - direction_balance
  - brent_close
  - wti_close
  - djt_close
  - djt_volume
  - brent_close_ffill
  - wti_close_ffill
  - djt_close_ffill
  - brent_pct_change
  - wti_pct_change
  - djt_pct_change
  - brent_wti_spread
  - brent_volatility_5d
  - is_trading_day
  - day_of_week
  - has_oil_post

Categorical features: 2
  - date
  - dominant_direction

Mixed types: PASS


In [11]:
# Save master dataset
master.to_csv('../data/processed/master_dataset.csv', index=False)
print(f'Saved master_dataset.csv: {len(master)} rows, {len(master.columns)} columns')

# Also save the cleaned post-level data (with timestamps and classifications)
posts.to_csv('../data/processed/posts_cleaned.csv', index=False)
print(f'Saved posts_cleaned.csv: {len(posts)} posts')

print('\nNext step: Run notebook 03 for oscillation pattern analysis.')

Saved master_dataset.csv: 448 rows, 25 columns
Saved posts_cleaned.csv: 21014 posts

Next step: Run notebook 03 for oscillation pattern analysis.


In [12]:
# Quick summary statistics
print('=== DESCRIPTIVE STATISTICS ===')
print()

trading_days = master[master['is_trading_day'] == 1]
print(f'Trading days: {len(trading_days)}')
print(f'Days with posts: {(master["post_count"] > 0).sum()}')
print(f'Days with oil-related posts: {(master["oil_post_count"] > 0).sum()}')
print()

print('Oil price summary (Brent):')
print(trading_days['brent_close'].describe().round(2))
print()

print('Daily return summary (Brent %):')
print(trading_days['brent_pct_change'].describe().round(2))

=== DESCRIPTIVE STATISTICS ===

Trading days: 305
Days with posts: 445
Days with oil-related posts: 414

Oil price summary (Brent):
count    305.00
mean      69.89
std        6.74
min       59.93
25%       65.40
50%       68.36
75%       72.54
max      103.23
Name: brent_close, dtype: float64

Daily return summary (Brent %):
count    304.00
mean       0.12
std        2.28
min       -7.01
25%       -1.27
50%        0.00
75%        1.27
max       12.53
Name: brent_pct_change, dtype: float64
